## Setup and Imports

In [8]:
del sys.modules['config.model_settings']

In [1]:
from pathlib import Path
import pandas as pd
import sys
from loguru import logger

sys.path.append(str(Path.cwd().parent / "src"))

from evaluation.labeled_evaluator import LabeledEvaluator
from evaluation.unlabeled_evaluator import UnlabeledEvaluator
from config.model_settings import MODEL_PATH, MODEL_CONFIG

## Load Configuration

In [2]:
# eval_config = TrainingConfig(Path("../config/evaluation_config.yaml"))

# # Paths
# LABELED_DATA_YAML = Path(eval_config.get('data.labeled_dataset_yaml'))
# UNLABELED_VIDEO_DIR = Path(eval_config.get('data.unlabeled_videos_dir'))
# OUTPUT_DIR = Path(eval_config.get('output.metrics_dir'))

# # Models to compare
# MODELS = {
#     "YOLOv8n (baseline)": eval_config.get('model.baseline_model'),
#     "Custom trained": Path(eval_config.get('model.custom_model')),
# }

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Unlabeled Test Videos

In [3]:
# video_loader = VideoLoader(UNLABELED_VIDEO_DIR)
# test_videos = video_loader.get_all_videos()

# print(f"Found {len(test_videos)} test videos:")
# for video in test_videos:
#     print(f"  - {video.name}")

## Evaluate on Labeled Data

In [4]:
# labeled_results = {}

# for model_name, model_path in MODELS.items():
#     print(f"\nEvaluating {model_name} on labeled data...")
    
#     evaluator = LabeledEvaluator(model_path)
#     evaluator.load_model()
    
#     metrics = evaluator.evaluate(LABELED_DATA_YAML)
#     labeled_results[model_name] = metrics
    
#     print(f"Results:")
#     for metric_name, value in metrics.items():
#         print(f"  {metric_name}: {value:.4f}")

## Evaluate on Unlabeled Data

In [2]:
unlabeled_results = {}

model_name = "yolov8n"
video_path = "E:/01_personal_project/input/preprocessed/20251026_155328.mp4"
logger.info(f"\nEvaluating {model_name} on unlabeled videos...")
    
evaluator = UnlabeledEvaluator(MODEL_PATH, MODEL_CONFIG)
metrics = evaluator.evaluate_single_video(
    video_path=video_path
)
unlabeled_results[model_name] = metrics

# print(f"Results:")
# for metric_name, value in metrics.items():
#     print(f"  {metric_name}: {value:.4f}")

2025-12-12 21:35:54.089 | INFO     | __main__:<module>:5 - 
Evaluating yolov8n on unlabeled videos...
2025-12-12 21:35:54.112 | INFO     | evaluation.unlabeled_evaluator:evaluate_single_video:113 - Evaluating video: E:/01_personal_project/input/preprocessed/20251026_155328.mp4
2025-12-12 21:35:54.339 | DEBUG    | evaluation.unlabeled_evaluator:evaluate_single_video:131 - Processed frame 0
2025-12-12 21:35:58.201 | DEBUG    | evaluation.unlabeled_evaluator:evaluate_single_video:131 - Processed frame 100
2025-12-12 21:36:02.128 | DEBUG    | evaluation.unlabeled_evaluator:evaluate_single_video:131 - Processed frame 200
2025-12-12 21:36:02.982 | INFO     | evaluation.unlabeled_evaluator:evaluate_single_video:134 - Computing metrics...
2025-12-12 21:36:02.983 | SUCCESS  | evaluation.unlabeled_evaluator:evaluate_single_video:136 - Evaluation complete: {'confidence_mean': 0.7235317334428534, 'confidence_std': 0.1292894720655712, 'confidence_min': 0.5014619827270508, 'confidence_max': 0.908716

## Combine and Compare Results

In [ ]:
# Combine labeled and unlabeled results
all_results = {}
for model_name in MODELS.keys():
    all_results[model_name] = {
        **{f"labeled_{k}": v for k, v in labeled_results[model_name].items()},
        **{f"unlabeled_{k}": v for k, v in unlabeled_results[model_name].items()}
    }

comparison_df = pd.DataFrame(all_results).T

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string())

# Save to CSV
csv_path = OUTPUT_DIR / "model_comparison.csv"
comparison_df.to_csv(csv_path)
print(f"\nComparison saved to: {csv_path}")

## Identify Best Model

In [ ]:
# Best model by labeled mAP50-95
if 'labeled_mAP50-95' in comparison_df.columns:
    best_labeled = comparison_df['labeled_mAP50-95'].idxmax()
    print(f"Best on labeled data (mAP50-95): {best_labeled}")
    print(f"  Score: {comparison_df.loc[best_labeled, 'labeled_mAP50-95']:.4f}")

# Best model by unlabeled avg confidence
if 'unlabeled_avg_confidence' in comparison_df.columns:
    best_confidence = comparison_df['unlabeled_avg_confidence'].idxmax()
    print(f"\nBest confidence on unlabeled: {best_confidence}")
    print(f"  Score: {comparison_df.loc[best_confidence, 'unlabeled_avg_confidence']:.4f}")

# Most consistent tracks
if 'unlabeled_avg_track_length' in comparison_df.columns:
    best_tracking = comparison_df['unlabeled_avg_track_length'].idxmax()
    print(f"\nBest tracking consistency: {best_tracking}")
    print(f"  Score: {comparison_df.loc[best_tracking, 'unlabeled_avg_track_length']:.4f}")